In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
def clean_name(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.replace("'", "").replace(",", "").replace("(", "").replace(")", "")
    return ' '.join(text.split())

# print(clean_name("Corn Flakes, NFS"))
# print(clean_name("Kellogg's Corn Flakes"))

In [3]:
fpx = pd.read_csv('https://raw.githubusercontent.com/Barabasi-Lab/GroceryDB/main/data/GroceryDB_training_dataset_SRFNDSS_2001_2018_NOVA123_multi_compositions_12Nutrients.csv')
print(fpx.shape)
fpx.info()

(10499, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10499 entries, 0 to 10498
Data columns (total 18 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Unnamed: 0                    10499 non-null  int64  
 1   Food code                     10499 non-null  int64  
 2   Main food description         10499 non-null  object 
 3   version SR                    5386 non-null   float64
 4   year FNDDS                    5113 non-null   float64
 5   novaclass                     10499 non-null  int64  
 6   Protein                       10499 non-null  float64
 7   Total Fat                     10499 non-null  float64
 8   Carbohydrate                  10499 non-null  float64
 9   Sugars, total                 10499 non-null  float64
 10  Fiber, total dietary          10499 non-null  float64
 11  Calcium                       10499 non-null  float64
 12  Iron                          10499 non-null  fl

In [4]:
fpx['name_clean']=fpx['Main food description'].map(clean_name) #apply the cleaning function to all the rows
print(fpx)

       Unnamed: 0  Food code  \
0               0       1042   
1               1       1043   
2               2       1044   
3               3       1046   
4               4       1048   
...           ...        ...   
10494       10494   94000100   
10495       10495   94000100   
10496       10496   94100100   
10497       10497   94300100   
10498       10498   94300100   

                                   Main food description  version SR  \
0      Cheese, pasteurized process, American, fortifi...        24.0   
1                   Cheese, pasteurized process, pimento        25.0   
2                     Cheese, pasteurized process, swiss        28.0   
3      Cheese food, pasteurized process, American, vi...        28.0   
4           Cheese spread, pasteurized process, American        28.0   
...                                                  ...         ...   
10494                                         Water, tap         NaN   
10495                                  

In [5]:
# checking for duplicate food code 
print(fpx['Food code'].duplicated().sum())
print(fpx['novaclass'].value_counts().sort_index())

2261
novaclass
1    3389
2     275
3    1863
4    4972
Name: count, dtype: int64


In [6]:
# check if any duplicated codes in conflict about their NOVA class
conflict = fpx.groupby('Food code')['novaclass'].nunique()
print((conflict>1).sum())

0


In [7]:
fpx=fpx.drop_duplicates(subset='Food code', keep='last')
print(len(fpx))
print(fpx['Food code'].duplicated().sum())


8238
0


In [8]:
print(fpx.isna().sum())
print(sorted(fpx['novaclass'].unique().tolist()))
print(len(fpx))

Unnamed: 0                         0
Food code                          0
Main food description              0
version SR                      3809
year FNDDS                      4429
novaclass                          0
Protein                            0
Total Fat                          0
Carbohydrate                       0
Sugars, total                      0
Fiber, total dietary               0
Calcium                            0
Iron                               0
Sodium                             0
Vitamin C                          0
Cholesterol                        0
Fatty acids, total saturated       0
Total Vitamin A                    0
name_clean                         0
dtype: int64
[1, 2, 3, 4]
8238


In [9]:
fpx.info()
print(fpx['novaclass'].value_counts().sort_index())

<class 'pandas.core.frame.DataFrame'>
Index: 8238 entries, 0 to 10498
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Unnamed: 0                    8238 non-null   int64  
 1   Food code                     8238 non-null   int64  
 2   Main food description         8238 non-null   object 
 3   version SR                    4429 non-null   float64
 4   year FNDDS                    3809 non-null   float64
 5   novaclass                     8238 non-null   int64  
 6   Protein                       8238 non-null   float64
 7   Total Fat                     8238 non-null   float64
 8   Carbohydrate                  8238 non-null   float64
 9   Sugars, total                 8238 non-null   float64
 10  Fiber, total dietary          8238 non-null   float64
 11  Calcium                       8238 non-null   float64
 12  Iron                          8238 non-null   float64
 13  Sodium 

In [10]:
def log_to_grams(x):
    x = np.asarray(x, dtype =float)
    return np.where (x > -19, np.exp(x), 0.0) # less than -19 means zero

#rename columns
fpx_cols = {
    'Protein':'protein_g',
    'Total Fat': 'fat_g',
    'Carbohydrate':'carb_g',
    'Sugars, total':'sugars_g',
    'Fiber, total dietary':'fiber_g',
    'Fatty acids, total saturated':'sat_fat_g',
    'Sodium': 'sodium_g'
}

# creating new columns for nutrients in gram
for old, new in fpx_cols.items():
    fpx[new]= log_to_grams(fpx[old])
print(fpx.columns.tolist())

['Unnamed: 0', 'Food code', 'Main food description', 'version SR', 'year FNDDS', 'novaclass', 'Protein', 'Total Fat', 'Carbohydrate', 'Sugars, total', 'Fiber, total dietary', 'Calcium', 'Iron', 'Sodium', 'Vitamin C', 'Cholesterol', 'Fatty acids, total saturated', 'Total Vitamin A', 'name_clean', 'protein_g', 'fat_g', 'carb_g', 'sugars_g', 'fiber_g', 'sat_fat_g', 'sodium_g']


In [11]:
# inspect for any sign of impossible value
macros = ['protein_g','fat_g','carb_g','sugars_g','fiber_g','sat_fat_g','sodium_g']
print(fpx[macros].describe().T.round(2))

            count   mean    std  min   25%    50%    75%     max
protein_g  8238.0   9.56   9.75  0.0  1.63   6.50  14.60   85.60
fat_g      8238.0  10.06  14.60  0.0  1.00   4.60  13.52  100.00
carb_g     8238.0  26.18  28.73  0.0  2.53  12.40  50.49  100.00
sugars_g   8238.0  10.87  16.64  0.0  0.10   3.73  13.00   99.91
fiber_g    8238.0   2.09   4.16  0.0  0.00   0.60   2.50   79.00
sat_fat_g  8238.0   3.33   5.51  0.0  0.20   1.44   4.12   91.30
sodium_g   8238.0   0.33   0.87  0.0  0.03   0.09   0.46   38.76


In [12]:
# run some checks to see if the value are within reasoble range
eps = 0.01
checks = {
    'out_of_range': ~((fpx[macros] >= -eps) & (fpx[macros]<=100 +eps)).all(axis=1),
    'sodium_over_40': fpx['sodium_g']> 40,
    'sugar_gt_carb': fpx['sugars_g']> fpx['carb_g'] * 1.05 + 5,
    'satfat_gt_fat': fpx['sat_fat_g']> fpx['fat_g'] * 1.05 +0.5,
    'macros_over_100': (fpx['protein_g']+fpx['fat_g']+fpx['carb_g'])>105,   
}

for name, mask in checks.items():
    print(f'{name:16s} {mask.sum()}')

bad = pd.concat(checks.values(), axis =1).any(axis=1)
print('rows falling any check:', bad.sum(), 'of', len(fpx))
print(fpx.loc[bad, ['Main food description'] + macros].head(20))

out_of_range     0
sodium_over_40   0
sugar_gt_carb    0
satfat_gt_fat    0
macros_over_100  0
rows falling any check: 0 of 8238
Empty DataFrame
Columns: [Main food description, protein_g, fat_g, carb_g, sugars_g, fiber_g, sat_fat_g, sodium_g]
Index: []


In [13]:
# rename nutrients columns
rename = {
    'Food code':'code',
    'Main food description': 'name_original',
    'novaclass': 'nova',
    'Protein': 'protein_log', 
    'Total Fat': 'fat_log', 
    'Carbohydrate': 'carb_log',
    'Sugars, total': 'sugars_log',
    'Fiber, total dietary': 'fiber_log',
    'Fatty acids, total saturated': 'sat_fat_log',
    'Sodium': 'sodium_log'
}

fpx_small = fpx[['name_clean']+ list(rename) + macros].copy()
fpx_small = fpx_small.rename(columns = rename)
print(fpx_small.shape)
fpx_small.head(5)

(8238, 18)


,name_clean,code,name_original,nova,protein_log,fat_log,carb_log,sugars_log,fiber_log,sat_fat_log,sodium_log,protein_g,fat_g,carb_g,sugars_g,fiber_g,sat_fat_g,sodium_g
0,cheese pasteurized process american fortified ...,1042,"Cheese, pasteurized process, American, fortifi...",4,2.897568,3.459152,1.308333,0.815365,-20.000000,2.893533,0.513422,18.13,31.79,3.70,2.26,0.0,18.057,1.671
1,cheese pasteurized process pimento,1043,"Cheese, pasteurized process, pimento",4,3.096934,3.440418,0.548121,-0.478036,-2.302585,2.978739,0.356275,22.13,31.20,1.73,0.62,0.1,19.663,1.428
2,cheese pasteurized process swiss,1044,"Cheese, pasteurized process, swiss",4,3.208017,3.219276,0.741937,0.207014,-20.000000,2.775397,0.314811,24.73,25.01,2.10,1.23,0.0,16.045,1.370
3,cheese food pasteurized process american vitam...,1046,"Cheese food, pasteurized process, American, vi...",4,2.824944,3.243764,2.147100,1.720979,-20.000000,2.711843,0.249980,16.86,25.63,8.56,5.59,0.0,15.057,1.284
4,cheese spread pasteurized process american,1048,"Cheese spread, pasteurized process, American",4,2.797891,3.055415,2.166765,1.990610,-20.000000,2.589792,0.485508,16.41,21.23,8.73,7.32,0.0,13.327,1.625


In [14]:
def to_log(grams):
    grams = np.asarray(grams, dtype=float)
    return np.where(grams > 0, np.log(grams), -20.0)
print(fpx_small.isna().sum().sum())
print(fpx_small['code'].is_unique)
print(np.allclose(to_log(fpx_small['protein_g']),fpx_small['protein_log']))

0
True
True


/tmp/ipykernel_16/3614251823.py:3: RuntimeWarning: divide by zero encountered in log
  return np.where(grams > 0, np.log(grams), -20.0)


In [15]:
fpx_small['carb_excl_fiber_g'] = (fpx['carb_g']-fpx['fiber_g']).clip(lower=0)
fpx_small['carb_excl_fiber_log']= fpx_small['carb_excl_fiber_g'].apply(to_log)

/tmp/ipykernel_16/3614251823.py:3: RuntimeWarning: divide by zero encountered in log
  return np.where(grams > 0, np.log(grams), -20.0)


In [16]:
fpx_small.to_csv('/kaggle/working/fpx_clean.csv', index = False)
